# IDK-1 — Step 3: Architecture

Define dan verifikasi arsitektur IDK-1 ~100M.

**Tidak ada file yang disimpan** — ini pure verification step.

Checklist:
- [ ] Model instantiate tanpa error
- [ ] Jumlah params: 100-110M
- [ ] Forward pass output shape benar
- [ ] Logit soft-capping bekerja

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

print(f"PyTorch : {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Config

In [ ]:
@dataclass
class IDK1Config:
    vocab_size  : int   = 40_000
    dim         : int   = 768
    n_layers    : int   = 12
    n_heads     : int   = 12
    n_kv_heads  : int   = 4        # GQA — lebih hemat VRAM saat inference
    ffn_dim     : int   = 2048     # SwiGLU hidden dim
    max_seq_len : int   = 1024
    dropout     : float = 0.0
    norm_eps    : float = 1e-5
    rope_theta  : float = 500_000  # LLaMA-3 style (bukan 10_000)
    logit_cap   : float = 30.0     # Gemma 2 style soft-capping

    @property
    def head_dim(self):
        return self.dim // self.n_heads


cfg = IDK1Config()
print("IDK1Config:")
for k, v in cfg.__dict__.items():
    print(f"  {k:<14} = {v}")
print(f"  {'head_dim':<14} = {cfg.head_dim}")

## 2. Building Blocks

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def precompute_rope(head_dim, seq_len, theta=10000.0, device="cpu"):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t     = torch.arange(seq_len, device=device)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    B, T, H, D = x.shape
    x1  = x[..., :D//2]
    x2  = x[..., D//2:]
    cos = cos[:T].unsqueeze(0).unsqueeze(2)
    sin = sin[:T].unsqueeze(0).unsqueeze(2)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


print("RMSNorm + RoPE OK")

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg: IDK1Config):
        super().__init__()
        self.n_heads    = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim   = cfg.head_dim
        self.groups     = cfg.n_heads // cfg.n_kv_heads  # = 3

        self.wq = nn.Linear(cfg.dim, cfg.n_heads    * cfg.head_dim, bias=False)
        self.wk = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wv = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wo = nn.Linear(cfg.n_heads * cfg.head_dim, cfg.dim,    bias=False)

    def forward(self, x, cos, sin):
        B, T, _ = x.shape

        q = self.wq(x).view(B, T, self.n_heads,    self.head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)

        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

        # Expand KV untuk match jumlah Q heads
        k = k.repeat_interleave(self.groups, dim=2)
        v = v.repeat_interleave(self.groups, dim=2)

        out = F.scaled_dot_product_attention(
            q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
            is_causal=True
        )
        return self.wo(out.transpose(1, 2).contiguous().view(B, T, -1))


print("GroupedQueryAttention OK")
print(f"  Q heads : {IDK1Config().n_heads}")
print(f"  KV heads: {IDK1Config().n_kv_heads}")
print(f"  Groups  : {IDK1Config().n_heads // IDK1Config().n_kv_heads}")

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, cfg: IDK1Config):
        super().__init__()
        self.gate = nn.Linear(cfg.dim, cfg.ffn_dim, bias=False)
        self.up   = nn.Linear(cfg.dim, cfg.ffn_dim, bias=False)
        self.down = nn.Linear(cfg.ffn_dim, cfg.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class TransformerBlock(nn.Module):
    def __init__(self, cfg: IDK1Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.attn      = GroupedQueryAttention(cfg)
        self.ffn_norm  = RMSNorm(cfg.dim, cfg.norm_eps)
        self.ffn       = SwiGLU(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x


print("SwiGLU + TransformerBlock OK")

## 3. Full Model

In [ ]:
class IDK1Model(nn.Module):
    def __init__(self, cfg: IDK1Config):
        super().__init__()
        self.cfg    = cfg
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.layers = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm   = RMSNorm(cfg.dim, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)

        # Weight tying — lm_head dan embed share weights
        # Hemat ~30M params dan biasanya improve performa
        self.lm_head.weight = self.embed.weight

        # Precompute RoPE dengan theta yang lebih besar (LLaMA-3 style)
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        x   = self.embed(idx)
        cos = self.rope_cos[:T]
        sin = self.rope_sin[:T]

        for layer in self.layers:
            x = layer(x, cos, sin)

        logits = self.lm_head(self.norm(x))

        # Logit soft-capping (Gemma 2) — cegah logit explode
        logits = self.cfg.logit_cap * torch.tanh(logits / self.cfg.logit_cap)

        return logits


print("IDK1Model OK")

## 4. Verifikasi

In [ ]:
# Instantiate
model = IDK1Model(cfg).to(device)

# Hitung params
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params    : {total_params/1e6:.2f}M")
print(f"Trainable params: {trainable_params/1e6:.2f}M")
print()

# Breakdown per komponen
embed_params = sum(p.numel() for p in model.embed.parameters())
layer_params = sum(p.numel() for p in model.layers.parameters())
norm_params  = sum(p.numel() for p in model.norm.parameters())

print(f"Embedding       : {embed_params/1e6:.2f}M  (shared dengan lm_head)")
print(f"Transformer ×{cfg.n_layers}  : {layer_params/1e6:.2f}M  ({layer_params/cfg.n_layers/1e6:.2f}M per layer)")
print(f"Final norm      : {norm_params/1e3:.1f}K")

assert 95e6 < total_params < 120e6, f"Params {total_params/1e6:.1f}M di luar range target (95-120M)!"
print(f"\nParam count CHECK PASSED")

In [ ]:
# Forward pass dengan dummy input
B, T = 2, 128   # batch=2, seq_len=128
dummy = torch.randint(0, cfg.vocab_size, (B, T), device=device)

with torch.no_grad():
    logits = model(dummy)

print(f"Input shape  : {dummy.shape}")
print(f"Output shape : {logits.shape}")
print(f"Expected     : [{B}, {T}, {cfg.vocab_size}]")

assert logits.shape == (B, T, cfg.vocab_size), "Output shape salah!"
print("\nForward pass CHECK PASSED")

In [ ]:
# Verifikasi logit soft-capping
logit_max = logits.max().item()
logit_min = logits.min().item()
logit_abs_max = logits.abs().max().item()

print(f"Logit max      : {logit_max:.4f}")
print(f"Logit min      : {logit_min:.4f}")
print(f"Logit |max|    : {logit_abs_max:.4f}")
print(f"Cap value      : {cfg.logit_cap}")

assert logit_abs_max <= cfg.logit_cap + 1e-4, f"Logit soft-capping tidak bekerja! max={logit_abs_max:.4f}"
print(f"\nLogit capping CHECK PASSED (semua logit dalam range [-{cfg.logit_cap}, {cfg.logit_cap}])")

In [ ]:
# Cek VRAM usage
if torch.cuda.is_available():
    torch.cuda.synchronize()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM allocated : {allocated:.2f} GB")
    print(f"VRAM reserved  : {reserved:.2f} GB")
    print(f"VRAM total     : {total:.1f} GB")
    print(f"VRAM headroom  : {total - reserved:.2f} GB")
    print()
    print("Note: ini model weights doang (float32 saat init).")
    print("Saat training float16 + batch=8 + seq=512, usage akan naik.")

## 5. Summary

In [ ]:
print("══════════════════════════════════")
print("     IDK-1 ARCHITECTURE VERIFIED  ")
print("══════════════════════════════════")
print(f"Params          : {total_params/1e6:.2f}M")
print(f"Architecture    : LLaMA-style decoder-only")
print(f"Attention       : GQA ({cfg.n_heads}Q / {cfg.n_kv_heads}KV heads)")
print(f"FFN             : SwiGLU (dim={cfg.ffn_dim})")
print(f"Norm            : RMSNorm")
print(f"Positional enc  : RoPE (theta={cfg.rope_theta:,.0f})")
print(f"Logit capping   : {cfg.logit_cap} * tanh(x / {cfg.logit_cap})")
print(f"Weight tying    : embed ↔ lm_head")
print()
print("Next: 04_pretrain.ipynb")